# Filipino Small Language Model — Tokenizer Experiment

This notebook trains a compact GPT-2–style language model on Filipino (Tagalog) text and compares two tokenizers:

| Tokenizer | Description |
|-----------|-------------|
| **Filipino Tokenizer** | `TagalogHFTokenizer` — morphology-aware BPE trained on Tagalog |
| **GPT-2 Tokenizer** | Standard English BPE baseline (`GPT2TokenizerFast`) |

**Hypothesis:** A tokenizer that understands Tagalog morphology should yield lower perplexity (better language modelling) than a general English tokenizer, given the same model architecture and training data.

---

**Corpus:** [Wikitext-TL-39](https://huggingface.co/datasets/jeypiic/wikitext-tl-39) — a Filipino Wikipedia text corpus  
**Model:** GPT-2 mini (6 layers, 6 heads, 384-dim embeddings, ~25 M params)  
**Hardware target:** Kaggle T4 GPU (~20 min for 1 epoch on 50 k samples)

## Install Dependencies

In [12]:
!pip install filipino-tokenizer[hf] torch plotly -q

## Configuration

> **This is the only cell you need to edit.**
>
> - Set `SKIP_TRAINING = True` to jump straight to the tokenizer analysis (Cell 14) without running training.
> - Update `CORPUS` to point to your local copy of `train_corpus.txt` if it differs from the Kaggle path.
> - All hyperparameters are pre-tuned for a **Kaggle T4 GPU**; training completes in approximately 20–30 minutes.
>
> **OOM fix:** `BATCH_SIZE` reduced from 64 → 16 and gradient accumulation added to preserve effective batch size of 64. `PYTORCH_ALLOC_CONF=expandable_segments:True` is set automatically in Cell 3 to reduce memory fragmentation.

In [13]:
SKIP_TRAINING = False

CORPUS = "/kaggle/input/datasets/jeypiic/wikitext-tl-39/train_corpus.txt"

# Model
VOCAB_SIZE  = 32_000
N_EMBD      = 384
N_LAYER     = 6
N_HEAD      = 6
MAX_LENGTH  = 256

# Training  ← tuned for T4 GPU, finishes in ~20-30 min
TRAIN_SPLIT       = 0.95
MAX_TRAIN_SAMPLES = 50_000   # enough for a valid comparison; full run takes 40+ hrs
BATCH_SIZE        = 16       # reduced from 64 to avoid OOM on T4 (14.6 GB VRAM)
GRAD_ACCUM_STEPS  = 4        # effective batch size = 16 * 4 = 64
LR                = 3e-4
EPOCHS            = 1        # 1 epoch on 50k samples is sufficient for perplexity comparison
OUTPUT_DIR        = "models/slm_filipino"
BASELINE_DIR      = "models/slm_gpt2tok"

print("Config loaded.")
print(f"  SKIP_TRAINING     = {SKIP_TRAINING}")
print(f"  MAX_TRAIN_SAMPLES = {MAX_TRAIN_SAMPLES:,}")
print(f"  BATCH_SIZE        = {BATCH_SIZE}  (effective = {BATCH_SIZE * GRAD_ACCUM_STEPS} with grad accum)")
print(f"  GRAD_ACCUM_STEPS  = {GRAD_ACCUM_STEPS}")
print(f"  EPOCHS            = {EPOCHS}")
print(f"  CORPUS            = {CORPUS}")

Config loaded.
  SKIP_TRAINING     = False
  MAX_TRAIN_SAMPLES = 50,000
  BATCH_SIZE        = 16  (effective = 64 with grad accum)
  GRAD_ACCUM_STEPS  = 4
  EPOCHS            = 1
  CORPUS            = /kaggle/input/datasets/jeypiic/wikitext-tl-39/train_corpus.txt


## Imports

In [14]:
import os
import re
import math
import shutil

# Reduce CUDA memory fragmentation (recommended when reserved >> allocated)
os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")

import torch
import numpy as np

from datasets import Dataset, load_from_disk
from transformers import (
    GPT2Config,
    GPT2LMHeadModel,
    GPT2TokenizerFast,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
)
from filipino_tokenizer.tagalog import TagalogHFTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cpu" and not SKIP_TRAINING:
    print("WARNING: Training on CPU will be extremely slow.")

Device: cuda


## Load Filipino Tokenizer

Loads `TagalogHFTokenizer` and applies a manual fix for a bug in v0.4.0 where special token IDs (`<pad>`, `<s>`, `</s>`) are not correctly wired up by the HF wrapper. The corrected tokenizer is then saved to `OUTPUT_DIR` for later reloading.

In [15]:
fil_tokenizer = TagalogHFTokenizer()

# Fix broken special token IDs (bug in TagalogHFTokenizer 0.4.0)
fil_tokenizer.pad_token_id = fil_tokenizer._inner.bpe.vocab["<pad>"]
fil_tokenizer.bos_token_id = fil_tokenizer._inner.bpe.vocab["<s>"]
fil_tokenizer.eos_token_id = fil_tokenizer._inner.bpe.vocab["</s>"]

os.makedirs(OUTPUT_DIR, exist_ok=True)
fil_tokenizer.save_pretrained(OUTPUT_DIR)

print(f"Filipino tokenizer loaded:")
print(f"  vocab_size : {fil_tokenizer.vocab_size:,}")
print(f"  pad_token  : {fil_tokenizer.pad_token!r} (id {fil_tokenizer.pad_token_id})")
print(f"  bos_token  : {fil_tokenizer.bos_token!r} (id {fil_tokenizer.bos_token_id})")
print(f"  eos_token  : {fil_tokenizer.eos_token!r} (id {fil_tokenizer.eos_token_id})")

Filipino tokenizer loaded:
  vocab_size : 32,000
  pad_token  : '<pad>' (id 0)
  bos_token  : '<s>' (id 2)
  eos_token  : '</s>' (id 3)


## Load Corpus and Train/Val Split

Reads `train_corpus.txt` from the Wikitext-TL-39 dataset. If the file is not found (e.g. running locally without the dataset), a small set of fallback Tagalog sentences is used instead so the rest of the notebook still runs end-to-end.

The corpus is capped at `MAX_TRAIN_SAMPLES` lines and split 95 / 5 into train and validation sets.

In [16]:
FALLBACK_LINES = [
    "Kumain siya ng pagkain sa hapagkainan.",
    "Ang mga bata ay masayang naglalaro sa labas ng bahay.",
    "Nagtatrabaho ang tatay sa opisina araw-araw.",
    "Pinakamahusay ang ginawa niya sa pagtatanghal.",
    "Inaprubahan ng Senado ang panukalang batas kahapon.",
    "Nakapagpapakain na ang nanay ng maraming bisita.",
    "Bumili ako ng mga prutas sa palengke kahapon.",
    "Maganda ang panahon ngayon kaya lumabas kami para maglakad.",
    "Nagbabasa ang mga estudyante ng libro sa silid-aklatan.",
    "Nagluluto ang nanay ng masarap na adobo para sa buong pamilya.",
] * 100

if os.path.isfile(CORPUS):
    with open(CORPUS, "r", encoding="utf-8") as f:
        all_lines = [l.strip() for l in f if l.strip()]
    print(f"Corpus: {len(all_lines):,} lines from Wikitext-TL-39")
else:
    all_lines = FALLBACK_LINES
    print(f"Corpus not found. Using {len(all_lines)} fallback lines.")

if MAX_TRAIN_SAMPLES and len(all_lines) > MAX_TRAIN_SAMPLES:
    all_lines = all_lines[:MAX_TRAIN_SAMPLES]
    print(f"Capped to {MAX_TRAIN_SAMPLES:,} lines")

split_idx   = int(len(all_lines) * TRAIN_SPLIT)
train_lines = all_lines[:split_idx]
val_lines   = all_lines[split_idx:]
print(f"Train: {len(train_lines):,} lines  |  Val: {len(val_lines):,} lines")

Corpus: 1,524,071 lines from Wikitext-TL-39
Capped to 50,000 lines
Train: 47,500 lines  |  Val: 2,500 lines


## Tokenization Helpers

Two tokenization functions are defined:

- **`tokenize_dataset_fil`** — calls the inner `TagalogTokenizer` directly, bypassing the broken `TagalogHFTokenizer.__call__` (which returns `None` for the `▁` boundary marker token in v0.4.0).
- **`tokenize_dataset_gpt2`** — standard HF batch call, works without workarounds.

Both pad sequences to `MAX_LENGTH` and return a HuggingFace `Dataset` with `input_ids` and `attention_mask` columns.

A `_prewarm` helper pre-populates the Filipino tokenizer's segmentation cache to speed up the first tokenization pass.

In [17]:
def _prewarm(tokenizer, lines):
    """Pre-warm the segmentation cache for faster tokenization."""
    inner = tokenizer._inner
    if hasattr(inner, "prewarm"):
        inner.prewarm(lines)
        return
    unique_words = set()
    for line in lines:
        for part in re.split(r'(\s+|[^\w])', line):
            if part and re.match(r'^\w+$', part):
                unique_words.add(part.lower())
    total = len(unique_words)
    print(f"Pre-warming cache: {total:,} unique words ...", end="\r")
    for word in unique_words:
        if word not in inner._segment_cache:
            inner._segment_cache[word] = inner._surface_annotate(word)
    print(f"Cache warmed: {total:,} unique words segmented.   ")


def tokenize_dataset_fil(lines, tokenizer, batch_size=5000):
    """
    Tokenize using the inner TagalogTokenizer directly, bypassing the
    broken TagalogHFTokenizer.__call__ (bug: _convert_token_to_id returns
    None for the ▁ boundary marker token in v0.4.0).
    """
    all_input_ids, all_attention_masks = [], []
    skipped = 0
    pad_id  = tokenizer._inner.bpe.vocab.get("<pad>", 0)

    for i in range(0, len(lines), batch_size):
        batch = lines[i : i + batch_size]
        for line in batch:
            if not line or not line.strip():
                skipped += 1
                continue
            try:
                ids = tokenizer._inner.encode(line)
                ids = ids[:MAX_LENGTH]
                mask = [1] * len(ids)
                pad_len = MAX_LENGTH - len(ids)
                ids  = ids  + [pad_id] * pad_len
                mask = mask + [0]      * pad_len
                all_input_ids.append(ids)
                all_attention_masks.append(mask)
            except Exception:
                skipped += 1
                continue

        pct = min(i + batch_size, len(lines)) / len(lines) * 100
        print(f"  {pct:5.1f}%  {min(i+batch_size, len(lines)):,}/{len(lines):,} lines", end="\r")

    print()
    if skipped:
        print(f"  ⚠ Skipped {skipped:,} lines")
    return Dataset.from_dict({"input_ids": all_input_ids, "attention_mask": all_attention_masks})


def tokenize_dataset_gpt2(lines, tokenizer, batch_size=512):
    """Tokenize using GPT-2 tokenizer (standard HF batch call works fine)."""
    all_input_ids, all_attention_masks = [], []

    for i in range(0, len(lines), batch_size):
        batch = [l for l in lines[i : i + batch_size] if l and l.strip()]
        if not batch:
            continue
        enc = tokenizer(
            batch,
            truncation=True,
            max_length=MAX_LENGTH,
            padding="max_length",
            return_attention_mask=True,
            return_token_type_ids=False,
            return_tensors=None,
        )
        all_input_ids.extend(enc["input_ids"])
        all_attention_masks.extend(enc["attention_mask"])
        pct = min(i + batch_size, len(lines)) / len(lines) * 100
        print(f"  {pct:5.1f}%  {min(i+batch_size, len(lines)):,}/{len(lines):,} lines", end="\r")

    print()
    return Dataset.from_dict({"input_ids": all_input_ids, "attention_mask": all_attention_masks})

## Build Filipino Tokenized Dataset

Tokenizes the train and validation splits using the Filipino tokenizer. Results are cached to disk so re-running the notebook skips the slow tokenization step.

In [18]:
FIL_TRAIN_CACHE = "tokenized_fil_train"
FIL_VAL_CACHE   = "tokenized_fil_val"

if os.path.isdir(FIL_TRAIN_CACHE) and os.path.isdir(FIL_VAL_CACHE):
    print("Loading cached Filipino-tokenized datasets...")
    train_ds = load_from_disk(FIL_TRAIN_CACHE)
    val_ds   = load_from_disk(FIL_VAL_CACHE)
    print("Loaded from cache.")
else:
    print("Pre-warming segmentation cache...")
    _prewarm(fil_tokenizer, all_lines)
    print("Tokenizing train split...")
    train_ds = tokenize_dataset_fil(train_lines, fil_tokenizer)
    print("Tokenizing val split...")
    val_ds   = tokenize_dataset_fil(val_lines, fil_tokenizer)
    print("Saving to disk...")
    train_ds.save_to_disk(FIL_TRAIN_CACHE)
    val_ds.save_to_disk(FIL_VAL_CACHE)
    print("Saved.")

print(f"\nTrain dataset: {len(train_ds):,} samples")
print(f"Val   dataset: {len(val_ds):,} samples")

Loading cached Filipino-tokenized datasets...
Loaded from cache.

Train dataset: 47,500 samples
Val   dataset: 2,500 samples


## Model Builder

Instantiates a GPT-2 mini model from scratch using `GPT2Config`. The same architecture is used for both the Filipino and GPT-2–tokenized models so that the only variable is the tokenizer.

A quick forward pass is run on 4 samples as a sanity check — the initial loss should be close to `log(vocab_size)` (random chance).

In [19]:
def build_model(tokenizer):
    config = GPT2Config(
        vocab_size   = tokenizer.vocab_size,
        n_embd       = N_EMBD,
        n_layer      = N_LAYER,
        n_head       = N_HEAD,
        n_positions  = MAX_LENGTH,
        n_ctx        = MAX_LENGTH,
        pad_token_id = tokenizer.pad_token_id,
        bos_token_id = tokenizer.bos_token_id,
        eos_token_id = tokenizer.eos_token_id,
    )
    return GPT2LMHeadModel(config)


model = build_model(fil_tokenizer)
total_params = sum(p.numel() for p in model.parameters())
print(f"Model: GPT-2 mini  |  {total_params/1e6:.1f}M params")
print(f"  Layers={N_LAYER}, Heads={N_HEAD}, Embd={N_EMBD}, MaxLen={MAX_LENGTH}")
print(f"  Vocab={fil_tokenizer.vocab_size:,}")

# Quick forward-pass sanity check
sample = train_ds[:4]
with torch.no_grad():
    out = model(
        input_ids=torch.tensor(sample["input_ids"]),
        labels=torch.tensor(sample["input_ids"]),
    )
print(f"\nForward pass OK  |  loss={out.loss.item():.4f}"
      f"  (expected ~{math.log(fil_tokenizer.vocab_size):.2f} at random init)")

Model: GPT-2 mini  |  23.0M params
  Layers=6, Heads=6, Embd=384, MaxLen=256
  Vocab=32,000

Forward pass OK  |  loss=10.3219  (expected ~10.37 at random init)


## Trainer

Wraps HuggingFace `Trainer` with a causal language modelling collator. Key training settings:

| Setting | Value | Note |
|---------|-------|------|
| Scheduler | Cosine | smoother convergence |
| Warmup | 200 steps | fixed steps (not ratio, which is deprecated) |
| `gradient_accumulation_steps` | 4 | effective batch = 16 × 4 = 64 |
| `gradient_checkpointing` | `True` | ~30% VRAM reduction, trades speed |
| `eval_strategy` | `"steps"` | replaces deprecated `evaluation_strategy` |
| `fp16` | `True` on CUDA | halves memory, speeds up training |
| `load_best_model_at_end` | `True` | keeps the checkpoint with lowest val loss |

In [20]:
def train_model(model, tokenizer, train_ds, val_ds, output_dir):
    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

    args = TrainingArguments(
        output_dir                  = output_dir,
        num_train_epochs            = EPOCHS,
        per_device_train_batch_size = BATCH_SIZE,
        per_device_eval_batch_size  = BATCH_SIZE,
        learning_rate               = LR,
        lr_scheduler_type           = "cosine",
        warmup_steps                = 200,      # fixed steps, not deprecated ratio
        weight_decay                = 0.01,
        gradient_accumulation_steps = GRAD_ACCUM_STEPS,
        gradient_checkpointing      = True,   # trades compute for ~30% less VRAM
        eval_strategy               = "steps",  # replaces deprecated evaluation_strategy
        eval_steps                  = 200,
        save_steps                  = 200,
        save_total_limit            = 2,
        load_best_model_at_end      = True,
        fp16                        = torch.cuda.is_available(),
        logging_steps               = 50,
        report_to                   = "none",
        dataloader_num_workers      = 2,
        dataloader_pin_memory       = torch.cuda.is_available(),
    )

    trainer = Trainer(
        model         = model,
        args          = args,
        train_dataset = train_ds,
        eval_dataset  = val_ds,
        data_collator = data_collator,
    )
    trainer.train()
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    return trainer

## Train Filipino Tokenizer Model

In [21]:
if not SKIP_TRAINING:
    print("Training Filipino-tokenized model...")
    trainer_fil = train_model(model, fil_tokenizer, train_ds, val_ds, OUTPUT_DIR)
    print(f"Saved to {OUTPUT_DIR}/")
else:
    print("SKIPPED — set SKIP_TRAINING = False to run.")

Training Filipino-tokenized model...


Step,Training Loss,Validation Loss
200,3.234728,3.202723


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to models/slm_filipino/


## Perplexity Evaluation

Perplexity is computed as `exp(mean cross-entropy loss)` over all non-padding tokens in the validation set. Lower perplexity indicates better next-token prediction.

In [22]:
def evaluate_perplexity(model, dataset, device="cpu", batch_size=16):
    model.eval()
    model.to(device)
    total_loss, total_tokens = 0.0, 0

    for i in range(0, len(dataset), batch_size):
        batch          = dataset[i : i + batch_size]
        input_ids      = torch.tensor(batch["input_ids"]).to(device)
        attention_mask = torch.tensor(batch["attention_mask"]).to(device)
        labels         = input_ids.clone()
        labels[attention_mask == 0] = -100

        with torch.no_grad():
            out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)

        n_tokens      = (labels != -100).sum().item()
        total_loss   += out.loss.item() * n_tokens
        total_tokens += n_tokens

    return math.exp(total_loss / total_tokens)


if not SKIP_TRAINING and os.path.isdir(OUTPUT_DIR):
    trained_model = GPT2LMHeadModel.from_pretrained(OUTPUT_DIR)
    ppl_fil = evaluate_perplexity(trained_model, val_ds, device=device)
    print(f"Filipino Tokenizer model — Perplexity: {ppl_fil:.2f}")
else:
    ppl_fil = None
    print("Skipped — no trained model found.")

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Filipino Tokenizer model — Perplexity: 24.79


## GPT-2 Baseline: Tokenize and Train

Repeats the full pipeline (dataset tokenization → model init → training → evaluation) using the standard GPT-2 tokenizer as a baseline. The GPT-2–tokenized dataset is also cached to disk.

In [23]:
gpt2_tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
gpt2_tokenizer.pad_token    = gpt2_tokenizer.eos_token
gpt2_tokenizer.pad_token_id = gpt2_tokenizer.eos_token_id
print(f"GPT-2 tokenizer vocab size: {gpt2_tokenizer.vocab_size:,}")

GPT2_TRAIN_CACHE = "tokenized_gpt2_train"
GPT2_VAL_CACHE   = "tokenized_gpt2_val"

if not SKIP_TRAINING:
    if os.path.isdir(GPT2_TRAIN_CACHE) and os.path.isdir(GPT2_VAL_CACHE):
        print("Loading cached GPT-2-tokenized datasets...")
        gpt2_train_ds = load_from_disk(GPT2_TRAIN_CACHE)
        gpt2_val_ds   = load_from_disk(GPT2_VAL_CACHE)
        print("Loaded from cache.")
    else:
        print("Tokenizing with GPT-2 tokenizer...")
        gpt2_train_ds = tokenize_dataset_gpt2(train_lines, gpt2_tokenizer)
        gpt2_val_ds   = tokenize_dataset_gpt2(val_lines,   gpt2_tokenizer)
        gpt2_train_ds.save_to_disk(GPT2_TRAIN_CACHE)
        gpt2_val_ds.save_to_disk(GPT2_VAL_CACHE)
        print("Saved.")

    print("Training GPT-2-tokenized model...")
    os.makedirs(BASELINE_DIR, exist_ok=True)
    gpt2_model = build_model(gpt2_tokenizer)
    trainer_gpt2 = train_model(gpt2_model, gpt2_tokenizer,
                               gpt2_train_ds, gpt2_val_ds, BASELINE_DIR)

    gpt2_trained = GPT2LMHeadModel.from_pretrained(BASELINE_DIR)
    ppl_gpt2 = evaluate_perplexity(gpt2_trained, gpt2_val_ds, device=device)
    print(f"GPT-2 Tokenizer model — Perplexity: {ppl_gpt2:.2f}")
else:
    ppl_gpt2 = None
    print("SKIPPED.")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

GPT-2 tokenizer vocab size: 50,257
Tokenizing with GPT-2 tokenizer...
  100.0%  47,500/47,500 lines
  100.0%  2,500/2,500 lines


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT-2 Tokenizer model — Perplexity: 100.38


## Results Comparison

Prints a side-by-side perplexity table and renders an interactive Plotly bar chart. The percentage difference indicates how much better (lower perplexity) the winning tokenizer performs.

In [42]:
if ppl_fil is not None and ppl_gpt2 is not None:
    import plotly.graph_objects as go
    from IPython.display import display, HTML

    improvement = (ppl_gpt2 - ppl_fil) / ppl_gpt2 * 100
    winner = "Filipino Tokenizer" if ppl_fil < ppl_gpt2 else "GPT-2 Tokenizer"

    print("=" * 50)
    print(f"{'Tokenizer':<25} {'Perplexity':>12}")
    print("-" * 50)
    print(f"{'Filipino Tokenizer':<25} {ppl_fil:>12.2f}")
    print(f"{'GPT-2 Tokenizer':<25} {ppl_gpt2:>12.2f}")
    print("-" * 50)
    print(f"Winner: {winner}  ({abs(improvement):.1f}% "
          f"{'lower' if ppl_fil < ppl_gpt2 else 'higher'} perplexity)")
    print("=" * 50)

    fig = go.Figure(go.Bar(
        x=["Filipino Tokenizer", "GPT-2 Tokenizer"],
        y=[ppl_fil, ppl_gpt2],
        marker_color=["#22c55e", "#6b7280"],
        text=[f"{ppl_fil:.2f}", f"{ppl_gpt2:.2f}"],
        textposition="outside",
    ))
    fig.update_layout(
        paper_bgcolor="#ffffff", plot_bgcolor="#ffffff",
        font=dict(family="Inter, sans-serif"),
        title="Perplexity on Wikitext-TL-39 validation set<br>"
              "<sup>Lower is better. Same model architecture, same data — only tokenizer differs.</sup>",
        yaxis_title="Perplexity (lower = better)",
        height=500,
    )

    display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))
else:
    print("Results not available — set SKIP_TRAINING = False and run training first.")

Tokenizer                   Perplexity
--------------------------------------------------
Filipino Tokenizer               24.79
GPT-2 Tokenizer                 100.38
--------------------------------------------------
Winner: Filipino Tokenizer  (75.3% lower perplexity)


## Tokenizer Analysis (No Training Required)

This cell runs independently of the training cells and provides a statistical comparison of the two tokenizers on 2,000 validation lines:

- **Fertility** — average number of tokens produced per whitespace word. Lower fertility means the tokenizer packs more meaning per token, leaving more room for context within `MAX_LENGTH`.
- **Mean sequence length** — average token count per line.
- **Context window utilisation** — what fraction of `MAX_LENGTH` is actually used on average.

An interactive Plotly figure shows both the fertility bar chart and the sequence-length distribution histogram side-by-side.

In [43]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

sample_lines = val_lines[:2000]


def analyze_tokenizer(lines, tokenizer, name, use_inner=False):
    total_tokens, total_words = 0, 0
    seq_lengths = []
    for line in lines:
        words = [w for w in line.split() if w]
        total_words += len(words)
        if use_inner:
            ids = tokenizer._inner.encode(line)
            ids = ids[:MAX_LENGTH]
        else:
            ids = tokenizer.encode(line, truncation=True, max_length=MAX_LENGTH)
        total_tokens += len(ids)
        seq_lengths.append(len(ids))
    return {
        "name":        name,
        "fertility":   total_tokens / max(total_words, 1),
        "seq_lengths": seq_lengths,
        "utilization": np.mean(seq_lengths) / MAX_LENGTH * 100,
        "mean_len":    np.mean(seq_lengths),
    }


print("Analyzing tokenizers on 2,000 val lines...")
fil_stats  = analyze_tokenizer(sample_lines, fil_tokenizer,  "Filipino Tok", use_inner=True)
gpt2_stats = analyze_tokenizer(sample_lines, gpt2_tokenizer, "GPT-2 Tok",    use_inner=False)

print(f"\n{'Metric':<30} {'Filipino Tok':>15} {'GPT-2 Tok':>15}")
print("-" * 62)
print(f"{'Fertility (tokens/word)':<30} {fil_stats['fertility']:>15.2f} {gpt2_stats['fertility']:>15.2f}")
print(f"{'Mean sequence length':<30} {fil_stats['mean_len']:>15.1f} {gpt2_stats['mean_len']:>15.1f}")
print(f"{'Context window utilization':<30} {fil_stats['utilization']:>14.1f}% {gpt2_stats['utilization']:>14.1f}%")

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=("Fertility (tokens/word)", "Sequence Length Distribution"))
fig.add_trace(go.Bar(
    x=["Filipino Tok", "GPT-2 Tok"],
    y=[fil_stats["fertility"], gpt2_stats["fertility"]],
    marker_color=["#22c55e", "#6b7280"],
    text=[f"{fil_stats['fertility']:.2f}", f"{gpt2_stats['fertility']:.2f}"],
    textposition="outside", showlegend=False,
), row=1, col=1)
fig.add_trace(go.Histogram(x=fil_stats["seq_lengths"],  name="Filipino Tok",
                           marker_color="#22c55e", opacity=0.7, nbinsx=40), row=1, col=2)
fig.add_trace(go.Histogram(x=gpt2_stats["seq_lengths"], name="GPT-2 Tok",
                           marker_color="#6b7280", opacity=0.7, nbinsx=40), row=1, col=2)
fig.update_layout(paper_bgcolor="#ffffff", plot_bgcolor="#ffffff",
                  font=dict(family="Inter, sans-serif"),
                  title="Tokenizer Comparison on Filipino Val Set",
                  barmode="overlay", height=500)
fig.show()

Analyzing tokenizers on 2,000 val lines...

Metric                            Filipino Tok       GPT-2 Tok
--------------------------------------------------------------
Fertility (tokens/word)                   2.53            2.05
Mean sequence length                      57.6            46.8
Context window utilization               22.5%           18.3%


## Trained Models Comparison

### Load models

In [26]:
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
from filipino_tokenizer.tagalog import TagalogHFTokenizer
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

# --- Filipino tokenizer model ---
fil_tokenizer_inf = TagalogHFTokenizer()
fil_tokenizer_inf.pad_token_id = fil_tokenizer_inf._inner.bpe.vocab["<pad>"]
fil_tokenizer_inf.bos_token_id = fil_tokenizer_inf._inner.bpe.vocab["<s>"]
fil_tokenizer_inf.eos_token_id = fil_tokenizer_inf._inner.bpe.vocab["</s>"]

fil_model_inf = GPT2LMHeadModel.from_pretrained(OUTPUT_DIR).to(device)
fil_model_inf.eval()

# --- GPT-2 tokenizer model ---
gpt2_tokenizer_inf = GPT2TokenizerFast.from_pretrained("gpt2")
gpt2_tokenizer_inf.pad_token    = gpt2_tokenizer_inf.eos_token
gpt2_tokenizer_inf.pad_token_id = gpt2_tokenizer_inf.eos_token_id

gpt2_model_inf = GPT2LMHeadModel.from_pretrained(BASELINE_DIR).to(device)
gpt2_model_inf.eval()

print(f"Both models loaded on {device}")

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Both models loaded on cuda


In [32]:
def generate_text(prompt, model, tokenizer, use_inner=False,
                  max_new_tokens=60, temperature=0.2, top_k=50,
                  top_p=0.95, repetition_penalty=1.2):
    if use_inner:
        ids = tokenizer._inner.encode(prompt)
        input_ids      = torch.tensor([ids], dtype=torch.long).to(device)
        attention_mask = torch.ones_like(input_ids)
    else:
        enc = tokenizer(prompt, return_tensors="pt")
        input_ids      = enc.input_ids.to(device)
        attention_mask = enc.attention_mask.to(device)

    prompt_len = input_ids.shape[1]

    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            attention_mask     = attention_mask,
            max_new_tokens     = max_new_tokens,
            do_sample          = True,
            temperature        = temperature,
            top_k              = top_k,
            top_p              = top_p,
            repetition_penalty = repetition_penalty,
            pad_token_id       = tokenizer.pad_token_id,
            eos_token_id       = tokenizer.eos_token_id,
        )

    new_ids = output_ids[0][prompt_len:].tolist()

    if use_inner:
        return tokenizer._inner.decode(new_ids).strip()
    else:
        return tokenizer.decode(new_ids, skip_special_tokens=True).strip()

In [38]:
def show_tokenization(text, fil_tok, gpt2_tok):
    fil_ids    = fil_tok._inner.encode(text)
    id_to_token = {v: k for k, v in fil_tok._inner.bpe.vocab.items()}
    fil_tokens = [id_to_token.get(i, f"<unk:{i}>") for i in fil_ids]

    gpt2_ids    = gpt2_tok.encode(text)
    gpt2_tokens = gpt2_tok.convert_ids_to_tokens(gpt2_ids)

    print(f"Text         : {text}")
    print(f"Filipino Tok ({len(fil_tokens):>3} tokens): {fil_tokens}")
    print(f"GPT-2 Tok    ({len(gpt2_tokens):>3} tokens): {gpt2_tokens}")
    print()

SAMPLE_TEXTS = [
    "kumain ka na ba?",
    "ano ulam?",
    "sabik na akong makita ka"
]

print("=" * 72)
print("TOKEN SEGMENTATION COMPARISON")
print("=" * 72)
for text in SAMPLE_TEXTS:
    show_tokenization(text, fil_tokenizer_inf, gpt2_tokenizer_inf)

TOKEN SEGMENTATION COMPARISON
Text         : kumain ka na ba?
Filipino Tok ( 12 tokens): ['k', '▁', 'um', '▁', 'ain', ' ', 'ka', ' ', 'na', ' ', 'ba', '?']
GPT-2 Tok    (  7 tokens): ['k', 'um', 'ain', 'Ġka', 'Ġna', 'Ġba', '?']

Text         : ano ulam?
Filipino Tok (  4 tokens): ['ano', ' ', 'ulam', '?']
GPT-2 Tok    (  4 tokens): ['ano', 'Ġu', 'lam', '?']

Text         : sabik na akong makita ka
Filipino Tok ( 11 tokens): ['sabik', ' ', 'na', ' ', 'akong', ' ', 'ma', '▁', 'kita', ' ', 'ka']
GPT-2 Tok    ( 10 tokens): ['s', 'ab', 'ik', 'Ġna', 'Ġak', 'ong', 'Ġm', 'ak', 'ita', 'Ġka']



In [36]:
PROMPTS = [
    "kumain ka na ba?",
    "ano ulam?",
    "sabik na akong makita ka"
]

SEP = "─" * 72

for prompt in PROMPTS:
    out_fil  = generate_text(prompt, fil_model_inf,  fil_tokenizer_inf,  use_inner=True)
    out_gpt2 = generate_text(prompt, gpt2_model_inf, gpt2_tokenizer_inf, use_inner=False)

    print(SEP)
    print(f"PROMPT      : {prompt}")
    print(f"Filipino Tok: {prompt} {out_fil}")
    print(f"GPT-2 Tok   : {prompt} {out_gpt2}")

print(SEP)

────────────────────────────────────────────────────────────────────────
PROMPT      : kumain ka na ba?
Filipino Tok: kumain ka na ba? ito saan. , ng mga bilang ang maiyang karing magtulad mga mai ng na ). iumsama pagi mula ang
GPT-2 Tok   : kumain ka na ba? ang mga pagpapipit sa pang paggamamag @-@ hindi ng taong kungalahilan. o nagmula at nito at ginag @-@ maaaring isangkakasikay ( magkupawa
────────────────────────────────────────────────────────────────────────
PROMPT      : ano ulam?
Filipino Tok: ano ulam? ang mga kainumian pa natapos ng pagnahon sa mga @-@ mga nagklat pangaumhin maan ay ng mga at
GPT-2 Tok   : ano ulam? @-@ ayon ang hindi nito nagpapagmakas sa hin.iya at iba pang dahatayo ni mga dalala sa isang kungkakamit na lahagi ng mga salihakasan
────────────────────────────────────────────────────────────────────────
PROMPT      : sabik na akong makita ka
Filipino Tok: sabik na akong makita ka an na mga lamaasaan ng bilaian.ang si tao mag pagpaang , mga ang noong mga ay mg